# Encoding Categorical Features for ML Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Encode categorical features for ML models
- Use label encoding
- Use one-hot encoding
- Apply encoding techniques to datasets
- Understand when to use each method

## 🔗 Where this fits

**Builds on:** Unit 2, lesson 06 "Developing Simple Supervised and Unsupervised Learning Models" — those models only accept numbers, so text columns must be encoded.

**Used later in:** Course 04 (AIAT 114) — Unit 1 and Course 05 (AIAT 115) — Unit 2, where encoding becomes part of a full preprocessing pipeline.

---

This notebook covers practical activities from **Course 01, Unit 2**:
- Encoding categorical features for ML models

---

## Introduction

**Categorical encoding** converts categorical data into numerical format that machine learning models can process, using techniques like label encoding and one-hot encoding.


## 🎯 A category that should never have been a feature

Between 2005 and 2019 the Dutch tax administration ran a risk-classification
system over families claiming childcare benefits. Among the attributes it used
was **nationality**. Roughly **26,000 families** were wrongly accused of fraud and
ordered to repay allowances in full — often €20,000 to €60,000, with penalties,
and no payment plan. Families with roots in the former Dutch colonies were hit
hardest. A parliamentary inquiry called the treatment an "unprecedented
injustice", and in **January 2021 the Dutch government resigned** over it.

That scandal is about many things — oversight, appeal rights, the presumption of
guilt — and one of them is the subject of this notebook. Nationality is a
**nominal** attribute: a set of names with no order and no distance between them.
The moment it enters a model it has to be turned into numbers, and the choice of
how carries an implicit claim. Encode it as `0, 1, 2, ...` and you have told a
linear or distance-based model that these categories lie on a line, that one is
"greater" than another and that some pairs are closer than others. None of that
is true, and the model will use it anyway.

### What goes wrong without a deliberate encoding choice

Two failures, in this order. **The silent one:** the model treats invented order
as real structure and learns relationships that exist only in your integer
mapping. **The loud one:** you avoid that by one-hot encoding a column with
thousands of categories and your feature matrix explodes. This notebook shows
both effects on five rows, where you can count them by hand.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Setup: a tiny DataFrame with three text columns. Models cannot consume strings, so
# this notebook shows the two standard ways to turn categories into numbers.
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

print("✅ Libraries imported!")
print("\nEncoding Categorical Features")
print("=" * 60)

# Sample categorical data
# Three categorical columns with different numbers of distinct values.
data = {
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size': ['small', 'large', 'medium', 'small', 'large'],
    'category': ['A', 'B', 'A', 'C', 'B']
}

df = pd.DataFrame(data)
print("\nOriginal data:")
print(df)
print(f"\nData types:\n{df.dtypes}")

print("\n✅ Data prepared!")

✅ Libraries imported!

Encoding Categorical Features

Original data:
   color    size category
0    red   small        A
1   blue   large        B
2  green  medium        A
3    red   small        C
4   blue   large        B

Data types:
color       object
size        object
category    object
dtype: object

✅ Data prepared!


In [2]:
# Label encoding maps each category to an integer (blue=0, green=1, ...). Compact, but
# it invents an order — only safe for truly ordinal data or tree models.
# Label Encoding - Assigns integer labels
print("=" * 60)
print("LABEL ENCODING")
print("=" * 60)

label_encoder = LabelEncoder()

# Encode each categorical column
# Encode every text column and show the category -> integer mapping side by side.
for col in df.select_dtypes(include=['object']).columns:
    df[f'{col}_encoded'] = label_encoder.fit_transform(df[col])
    print(f"\n{col} encoding:")
    print(f"  Original: {df[col].unique()}")
    print(f"  Encoded: {df[f'{col}_encoded'].unique()}")

print("\n✅ Label encoding completed!")

LABEL ENCODING

color encoding:
  Original: ['red' 'blue' 'green']
  Encoded: [2 0 1]

size encoding:
  Original: ['small' 'large' 'medium']
  Encoded: [2 0 1]

category encoding:
  Original: ['A' 'B' 'C']
  Encoded: [0 1 2]

✅ Label encoding completed!


In [3]:
# One-hot encoding creates one binary column per category: no fake order, but the
# column count grows with the number of categories — the standard choice for nominal data.
# One-Hot Encoding - Creates binary columns
print("=" * 60)
print("ONE-HOT ENCODING")
print("=" * 60)

# Using pandas get_dummies
df_onehot = pd.get_dummies(df[['color', 'size', 'category']], prefix=['color', 'size', 'category'])
print("\nOne-hot encoded data:")
print(df_onehot)

# Compare column counts: 3 text columns became 11 binary ones — the cost of one-hot.
print(f"\nOriginal columns: {df[['color', 'size', 'category']].shape[1]}")
print(f"One-hot columns: {df_onehot.shape[1]}")

print("\n✅ One-hot encoding completed!")

ONE-HOT ENCODING

One-hot encoded data:
   color_blue  color_green  color_red  size_large  size_medium  size_small  \
0       False        False       True       False        False        True   
1        True        False      False        True        False       False   
2       False         True      False       False         True       False   
3       False        False       True       False        False        True   
4        True        False      False        True        False       False   

   category_A  category_B  category_C  
0        True       False       False  
1       False        True       False  
2        True       False       False  
3       False       False        True  
4       False        True       False  

Original columns: 3
One-hot columns: 9

✅ One-hot encoding completed!


## 📊 What the two runs actually showed

Same five rows, same three text columns, one thing changed — the encoder:

| | what a category becomes | resulting columns | order implied? |
|---|---|---|---|
| **Label encoding** | one integer (`blue`→0, `green`→1, `red`→2) | **3** | **yes — and it is fictional** |
| **One-hot encoding** | one binary column per category | **9** | no |

**The conclusion these outputs support:** the two encoders carry exactly the same
information and make opposite trade-offs. Label encoding kept the table at 3
columns and inserted a claim nobody verified — that `red` (2) is twice `green` (1)
and further from `blue` (0). One-hot removed the claim and tripled the width, from
3 columns to 9, on a toy with only three categories per column.

Now scale it. A `city` column for Saudi Arabia has hundreds of values; a product
SKU column has tens of thousands. One-hot turns each into that many columns.
That is where this trade stops being an exercise.

**One thing the printout does not show:** `size` is `small / medium / large` —
genuinely ordered. Label encoding is the *right* choice there, and it happened to
map them 2 / 1 / 0, i.e. backwards, because `LabelEncoder` sorts alphabetically
and knows nothing about size. Correct method, wrong order, no error message.


## 💬 Discuss

1. Of our three columns — `color`, `size`, `category` — exactly one has a real
   order. **Which encoder would you choose for each, and how would you fix the
   alphabetical ordering `LabelEncoder` imposed on `size`?** Write the mapping you
   would use.
2. At prediction time a new row arrives with `color = "yellow"`, a value that was
   never in the training data. Trace what each encoder does. Which failure would
   you rather debug at 2 a.m., and what would you have built to make it loud?
3. **Regional practice:** you are building a loan model for a Saudi bank and the
   data includes `nationality` and `city_of_residence`. For each, argue whether it
   belongs in the model at all — and if it does, how you would encode it and what
   you would monitor afterwards. Assume someone will ask you to justify a refusal
   to a customer.


## Summary

This notebook covered:
- ✅ **Label Encoding**: Assigns integer labels to categories
- ✅ **One-Hot Encoding**: Creates binary columns for each category
- ✅ **When to use**: Label encoding for ordinal data, one-hot for nominal data

Categorical encoding is essential for preparing data for machine learning models.

## ⚠️ Where this breaks

- **Label encoding invents an order and most models believe it.** Linear and
  logistic regression, SVMs, k-nearest neighbours and neural networks all treat
  the integers as magnitudes: `red = 2` really is "twice" `green = 1` to them.
  Tree-based models are the exception — they only ever split on thresholds, so an
  arbitrary integer mapping mostly costs them a few extra splits. **The rule:**
  label-encode only for tree models, or for genuinely ordinal columns with the
  order set by hand.
- **`LabelEncoder` sorts alphabetically.** `large / medium / small` → 0 / 1 / 2 is
  the reverse of the real order, and nothing tells you. For ordinal columns, use
  an explicit mapping (`{'small': 0, 'medium': 1, 'large': 2}`) or
  `OrdinalEncoder(categories=[...])` where you supply the sequence.
- **One-hot encoding scales with cardinality, not with rows.** Hundreds of
  categories means hundreds of columns, most of them almost always zero — which
  costs memory, slows training, and gives linear models very little data per
  coefficient. **Cheaper alternatives:** group rare categories into "other";
  target/mean encoding with careful cross-fitted folds; hashing; or learned
  embeddings for very high cardinality.
- **Unseen categories break both encoders at prediction time.** `LabelEncoder`
  raises on a value it has not seen; `pd.get_dummies` silently produces a
  different set of columns than training did, which is worse. **Use instead:**
  `sklearn.preprocessing.OneHotEncoder(handle_unknown='ignore')` inside a
  `Pipeline`, so training and serving share one fitted object.
- **`get_dummies` fitted on the test set is a bug factory.** Run it separately on
  train and test and you get different columns whenever a category is missing from
  one of them. Fit the encoder once, on training data, and apply it.
- **The dummy-variable trap.** For a plain linear regression with an intercept,
  *k* one-hot columns are perfectly collinear; drop one (`drop_first=True`).
  Regularised models and trees do not care.
- **Encoding cannot make a variable appropriate.** The Dutch system's failure was
  not that nationality was encoded badly. It was that nationality was in the model
  at all. Choose the columns before you choose the encoder.


## 📚 References

1. Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825–2830. <https://arxiv.org/abs/1201.0490>
2. Micci-Barreca, D. (2001). *A Preprocessing Scheme for High-Cardinality Categorical Attributes in Classification and Prediction Problems*. ACM SIGKDD Explorations, 3(1), 27–32.
3. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.), Ch. 3 (qualitative predictors). Springer.